# Term Lookup Benchmarking

This notebook demonstrates how to benchmark SNOMED CT term search functionality using **real SNOMED concepts**.

## Overview

Term lookup evaluates how well a method can match clinical terms to their corresponding SNOMED CT concepts.

### Key Metrics:
- **Recall@K**: Whether the expected CUI appears in top-K results (binary: 1 or 0)
- **Precision@K**: Similarly computed
- **MRR (Mean Reciprocal Rank)**: Average of reciprocal ranks where expected CUI is found
- **Hit Rate**: Fraction of queries where expected CUI appears anywhere in results

### Data Source:
Uses actual SNOMED CT concepts from the UK Clinical terminology (102K+ active concepts).


In [ ]:
# Import required modules
from snomed_methods import create_term_lookup_from_directory
from snomed_methods.benchmarking.term import evaluate_term_lookup

## Load Real SNOMED Concept Data

We use actual SNOMED CT concepts instead of synthetic data.


In [ ]:
# Load real SNOMED terminology lookup
snomed_dir = "uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z"
lookup = create_term_lookup_from_directory(snomed_dir)

# Get random sample of real SNOMED concepts for benchmarking
import random

random.seed(42)

# Sample 100 concepts from the full terminology
sample_cuis = random.sample(list(lookup.concept_ids), min(100, len(lookup.concept_ids)))

In [ ]:
# Create benchmark dataset from real SNOMED concepts
dataset = []

for cui in sample_cuis:
    info = lookup.getconcept_info(cui)
    term = info.get("preferred_name", "")

    if term:  # Only include concepts with valid names
        dataset.append(
            {
                "term": term,
                "expected_cui": cui,
            }
        )

print(f"Dataset size: {len(dataset)}")
print("\nFirst 3 samples:")
for i, sample in enumerate(dataset[:3]):
    print(f"\nSample {i+1}:")
    print(f"  Term: '{sample['term']}'")
    print(f"  Expected CUI: {sample['expected_cui']}")

## Evaluate Real SNOMED Term Lookup

Now we use the actual `SnomedTermLookup.find_concepts_by_term()` method.


In [ ]:
# Real term lookup function using SnomedTermLookup
def real_term_lookup(term: str, top_n: int = 20) -> list:
    """Look up a clinical term in SNOMED CT."""
    results = lookup.find_concepts_by_term(term, top_n=top_n)
    return [(cui, name) for cui, name in results]

In [ ]:
# Evaluate real term lookup on SNOMED concepts
results = evaluate_term_lookup(
    lookup_func=real_term_lookup,
    dataset=dataset[:50],  # First 50 samples for demo
    k_values=[1, 3, 5, 10, 20],
)

In [ ]:
# Display benchmark results
print("\n=== Term Lookup Benchmark Results (Real SNOMED Data) ===")
for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    elif isinstance(value, int):
        print(f"{metric}: {value}")

## Rank Position Analysis

Analyze how often the expected CUI appears at specific positions.


In [ ]:
# Exact match rates at different ranks
print("\nExact match rates at different ranks:")
for pos in [0, 1, 2, 3]:
    key = f"exact_match@pos_{pos}"
    if key in results:
        print(f"Position {pos}: {results[key]:.4f}")

## Summary Statistics


In [ ]:
# Summary statistics
print("\nSummary Statistics:")
hit_rate = results.get("hit_rate", 0)
total_samples = results["num_samples"]
found_count = int(total_samples * hit_rate) if total_samples > 0 else 0
print(f"Hit Rate: {hit_rate:.4f} ({found_count}/{total_samples} queries found)")

mrr = results.get("mrr", 0)
print(f"MRR: {mrr:.4f}")

if mrr > 0:
    avg_rank = 1.0 / mrr
    print(f"Average rank for hits: {avg_rank:.2f}")